In [4]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import ast
from Bio.Seq import Seq
from Bio import SeqIO
import re
import matplotlib.pyplot as plt
from scipy.spatial import distance
import collections

In [5]:
def add_feature(existing_features, new_feature):
    return existing_features + (new_feature,)

In [6]:
phylogeneticOrder=()
phylDF = pd.read_csv("/LeeLab/HPRC/chromosomeY/HPRC_HGSVC3_sample_annotations_20March2025.txt",sep='\t')
for row in phylDF.index:
    if '/' in phylDF.at[row,'sample']:
        for sample in phylDF.at[row,'sample'].split("/"):
             phylogeneticOrder = add_feature(phylogeneticOrder, sample)
    else:
        phylogeneticOrder = add_feature(phylogeneticOrder, phylDF.at[row,'sample'])

haplogroups={}
for row in phylDF.index:
    if '/' in phylDF.at[row,'sample']:
        for sample in phylDF.at[row,'sample'].split("/"):
            haplogroups[sample]=phylDF.at[row,'haplogroup_ISOGG_v15.73']
    else:
        haplogroups[phylDF.at[row,'sample']]=phylDF.at[row,'haplogroup_ISOGG_v15.73']

haplogroups['NA12877']='R1b1a1b1a1a2e2'
haplogroups['NA12882']='R1b1a1b1a1a2e2'
haplogroups['NA12883']='R1b1a1b1a1a2e2'
haplogroups['NA12884']='R1b1a1b1a1a2e2'
haplogroups['NA12886']='R1b1a1b1a1a2e2'
haplogroups['200080']='R1b1a1b1a1a1c2b2b1a'
haplogroups['200084']='R1b1a1b1a1a1c2b2b1a'
haplogroups['200085']='R1b1a1b1a1a1c2b2b1a'

In [7]:
colorDict={
    'blue1':'#010f8e',
    'blue2':'#1307b3',
    'blue3':'#0000ff',
    'blue4':'#4f78ff',
    'teal1':'#00d6e5',
    'teal2':'#00eeff',
    'green1':'#00630d',
    'green2':'#00a115',
    'green3':'#00ff21',
    'red1':'#630002',
    'red2':'#940003',
    'red3':'#ff0005',
    'red4':'#ff4a4e',
    'gray1':'#808080',
    'gray2':'#bababa',
    'yellow1':'#d6d609',
    'yellow2':'#ffff00',
    'spacer1':'white',
    'spacer2':'white',
    'spacer3':'white'}

In [8]:
arangAnnotationDF = pd.read_csv('/LeeLab/HPRC/chromosomeY/Data/ArangAnnotations/HG00126_chrY.bed', header=None, sep='\t')

In [11]:
aanottDirectory = '/LeeLab/HPRC/chromosomeY/Data/ArangAnnotations/'
directory='/LeeLab/HPRC/chromosomeY/Data/ColorBlockDataframes_refined/'
for file in os.listdir(directory):
    if '.csv' in file:
        #print(file)
        sampleName = str(file.split("_")[0])
        df = pd.read_csv(directory+file).drop(columns=['Unnamed: 0'])
        
        if 'HG03456' in file:
            sampleName2 = str(file.split("_")[1])+"_"+str(file.split("_")[2])+"_"+str(file.split("_")[3])+".bed"
            arangAnnotationDF = pd.read_csv(aanottDirectory+sampleName2, header=None, sep='\t')

        else:
            arangAnnotationDF = pd.read_csv(aanottDirectory+sampleName+"_chrY.bed", header=None, sep='\t')
            
        arangAnnotationDF[1]=arangAnnotationDF[1].astype(int)
        arangAnnotationDF[2]=arangAnnotationDF[2].astype(int)
        testCoords = []
        for x,y in zip(df['RefinedStart'],df['RefinedEnd']):
            testCoords.append(x)
            testCoords.append(y)
        coordinateList = [(min(testCoords), max(testCoords))]
        mask = pd.Series(False, index=arangAnnotationDF.index)
        for coord_start, coord_end in coordinateList:
            mask |= (
                (arangAnnotationDF[1] <= coord_end) &
                (arangAnnotationDF[2] >= coord_start)
            )
        
        arangAnnotationDF2 = arangAnnotationDF[mask].copy()
        #arangAnnotationDF2 = arangAnnotationDF1[arangAnnotationDF1[3].isin(df['RefinedClassification'])].copy()
        df['ArangAnnotations']='NONE'
        df['ArangCoordinates']='NONE'
        for row in df.index:
            start = int(df.at[row,'RefinedStart'])
            end = int(df.at[row,'RefinedEnd'])
            matches=[]
            coordinates={}
            for arow in arangAnnotationDF2.index:
                if end >= arangAnnotationDF2.at[arow,1] and start <= arangAnnotationDF2.at[arow,2]:
                    matches.append(arangAnnotationDF2.at[arow,3]+arangAnnotationDF2.at[arow,5])
                    coordinates[arangAnnotationDF2.at[arow,3]] = str(arangAnnotationDF2.at[arow,1])+"-"+str(arangAnnotationDF2.at[arow,2])
                else:
                    continue
            df.at[row,'ArangAnnotations']='_'.join(matches)
            df.at[row,'ArangCoordinates']=coordinates

        
        comparisonFindings=[]
        coordinateAppendList=[]
        for mycolor, myorientation, arangColors, arangDict in zip(df['RefinedClassification'], df['RefinedOrientation'], df['ArangAnnotations'], df['ArangCoordinates']):
            flag=0
            comparisonCoordinateHits=[]
            orientDict={'sense':'+', 'antisense':'-'}
            if 'spacer' in mycolor:
                flag+=1
            
            elif 'plus' in mycolor or 'IR1' in mycolor:
                shortenedColor = mycolor.split("-")[0][:-1]
                for aColor in arangColors.split("_"):
                    if shortenedColor in aColor and aColor[-1] == orientDict[myorientation]:
                        comparisonCoordinateHits.append(aColor[:-1]+":"+arangDict[aColor[:-1]])
                        flag+=1
                            
                    else:
                        continue
            else:
                shortenedColor = mycolor[:-1]
                for aColor in arangColors.split("_"):
                    if shortenedColor in aColor and aColor[-1] == orientDict[myorientation]:
                        comparisonCoordinateHits.append(aColor[:-1]+":"+arangDict[aColor[:-1]])
                        flag+=1
                    else:
                        continue
            
            if flag==0:
                comparisonFindings.append("Bad")
            else:
                comparisonFindings.append("Good")

            coordinateAppendList.append(set(comparisonCoordinateHits))
            
        df['AnnotationComparison']=comparisonFindings
        df['AnnotationComparisonHits']=coordinateAppendList
        print(sampleName)
        print(collections.Counter(df['AnnotationComparison']))

        columnsList = ['Haplotype', 'RefinedLength','RefinedStart', 'RefinedEnd',
               'RefinedClassification',
               'RefinedOrientation','GAP_TEST', 'ArangAnnotations','AnnotationComparisonHits','AnnotationComparison']
        df['RefinedLength']=[x-y for x,y in zip(df['RefinedEnd'], df['RefinedStart'])]
        df2 = df[columnsList].copy()
        #df2.to_csv('/LeeLab/HPRC/chromosomeY/Data/ColorBlockComparisonArang/'+file.split(".pipeline")[0]+".wArang.csv")

    else:
        continue

NA18534
Counter({'Good': 28})
HG02071
Counter({'Good': 29})
HG03579
Counter({'Good': 29})
HG01952
Counter({'Good': 29})
HG01934
Counter({'Good': 29})
NA18879
Counter({'Good': 29})
HG01255
Counter({'Good': 29})
HG02055
Counter({'Good': 29})
NA19650
Counter({'Good': 29})
HG01106
Counter({'Good': 29})
HG03742
Counter({'Good': 28})
NA18989
Counter({'Good': 29})
HG01530
Counter({'Good': 28})
HG00126
Counter({'Good': 29})
HG02647
Counter({'Good': 28})
HG03065
Counter({'Good': 39})
HG02602
Counter({'Good': 30})
HG01258
Counter({'Good': 29})
NA20752
Counter({'Good': 29})
HG01167
Counter({'Good': 29})
HG005
Counter({'Good': 19})
HG01928
Counter({'Good': 29})
HG02698
Counter({'Good': 21})
HG003
Counter({'Good': 18})
HG03710
Counter({'Good': 28})
HG04187
Counter({'Good': 33})
HG01433
Counter({'Good': 11})
HG01252
Counter({'Good': 29})
HG03017
Counter({'Good': 29})
NA18952
Counter({'Good': 19})
HG02129
Counter({'Good': 19})
NA18983
Counter({'Good': 19})
HG03732
Counter({'Good': 29})
HG03225
Counte

In [ ]:
#Occurs in an aseembly with gaps so I am not worried. 
HG02015
Counter({'Good': 27, 'Bad': 1})